In [1]:
import json, math, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

sys.path.insert(0, "kev")
from models.kev_pointer import KevPointerModel

def make_one_hot(row):
    codes = row["option_codes"]
    if len(codes) != len(row["option_texts"]):
        raise ValueError(f"{row['qid']}: option_codes ({len(codes)}) != option_texts ({len(row['option_texts'])})")
    idx = codes.index(row["answer_code"])  # raises ValueError if answer_code not in option_codes
    vec = np.zeros(len(codes), dtype=np.float64)
    vec[idx] = 1.0
    return vec

In [2]:
COUNTRY_NAMES = {"US": "United States"}
AGE_BINS = [17, 29, 49, 64, 120]
AGE_LABELS = ["18-29", "30-49", "50-64", "65+"]
splits_by_year = json.loads(Path("splits/split.json").read_text())

frames = []
for year, splits in splits_by_year.items():
    df = pd.read_parquet(f"data/issp{year}/issp{year}_US_long.parquet")
    df = df[df["v"].isin(splits["test"])].copy()
    df["year"] = year
    df["qid"] = year + ":" + df["v"]
    frames.append(df)

test_df = pd.concat(frames, ignore_index=True)
test_df["c_alphan"] = [COUNTRY_NAMES[c] for c in test_df["c_alphan"]]
age_num = pd.to_numeric(test_df["AGE"], errors="raise")
age_bucketed = pd.cut(age_num, bins=AGE_BINS, labels=AGE_LABELS).astype(object)
if int(age_bucketed.isna().sum()) != int(test_df["AGE"].isna().sum()):
    raise ValueError("Some non-missing AGE values fell outside AGE_BINS")
test_df["AGE"] = age_bucketed
for col in ("option_codes", "option_texts"):
    test_df[col] = test_df[col].map(list)

test_df["one_hot"] = [make_one_hot(row) for _, row in test_df.iterrows()]

question_groups = {qid: q_df for qid, q_df in test_df.groupby("qid", sort=False)}
print(f"Test set: {len(test_df):,} answers across {len(question_groups)} questions")
test_df.head(2)

Test set: 92,992 answers across 72 questions


,CASEID,c_alphan,SEX,AGE,EDULEVEL,RELIGGRP,ATTEND,PARTY_LR,VOTE_LE,URBRURAL,...,question,type,reversed,option_codes,option_texts,answer_code,answer_text,year,qid,one_hot
0,2015008400000003,United States,Male,65+,College graduate/some postgrad,Catholic,Once a week,Conservative,Yes,The suburbs or outskirts of a big city,...,Please show how important you personally think...,score,True,"[1, 2, 3, 4, 5]","[Very important, Important, Neither important ...",2,Important,2015,2015:v11,"[0.0, 1.0, 0.0, 0.0, 0.0]"
1,2015008400000005,United States,Female,50-64,College graduate/some postgrad,Atheist,Never,Liberal,Yes,The suburbs or outskirts of a big city,...,Please show how important you personally think...,score,True,"[1, 2, 3, 4, 5]","[Very important, Important, Neither important ...",2,Important,2015,2015:v11,"[0.0, 1.0, 0.0, 0.0, 0.0]"


In [3]:
PERSONA_NAMES = {
    "c_alphan": "Country",
    "SEX": "Sex",
    "AGE": "Age",
    "EDULEVEL": "Education",
    "RELIGGRP": "Religion",
    "ATTEND": "Attendance of religious services",
    "PARTY_LR": "Political ideology",
    "VOTE_LE": "Voted in the last general election",
    "URBRURAL": "Place of living",
    "MARITAL": "Marital status",
    "MAINSTAT": "Main activity",
    "UNION": "Trade union membership",
    "US_PRTY": "Political party",
    "US_INC": "Income",
    "US_REG": "Region",
    "US_ETHN1": "Ancestry",
}
persona_cols = list(test_df.columns[1:test_df.columns.get_loc("v")])
if persona_cols != list(PERSONA_NAMES.keys()):
    raise ValueError(f"Persona columns mismatch: {persona_cols} != {list(PERSONA_NAMES.keys())}")

In [4]:
for k in PERSONA_NAMES.keys():
    print(test_df[k].value_counts())
    print('+'*50)

c_alphan
United States    92992
Name: count, dtype: int64
++++++++++++++++++++++++++++++++++++++++++++++++++
SEX
Female    50989
Male      41855
Name: count, dtype: int64
++++++++++++++++++++++++++++++++++++++++++++++++++
AGE
30-49    31462
50-64    23190
65+      22585
18-29    13016
Name: count, dtype: int64
++++++++++++++++++++++++++++++++++++++++++++++++++
EDULEVEL
Upper secondary                   40937
College graduate/some postgrad    34899
Less than high school              8614
Short-cycle tertiary               8305
Name: count, dtype: int64
++++++++++++++++++++++++++++++++++++++++++++++++++
RELIGGRP
Protestant               40349
Atheist                  23888
Catholic                 19501
Other Religions           2084
Other Christian           1941
Jewish                    1640
Buddhist                   771
Muslim                     689
Hindu                      667
Orthodox                   558
Other Asian Religions       79
Name: count, dtype: int64
+++++++++++++++

In [5]:
sample_question_df = question_groups['2016:v34']
sample_row = sample_question_df.iloc[0]
# sample_question_df
print(sample_question_df['question'].iloc[0])
print(sample_question_df['option_texts'].iloc[0])

for sex, subset in sample_question_df.groupby('SEX'):
    marginal_dist = np.stack(subset['one_hot'].values).mean(axis=0)
    print(sex, len(subset), marginal_dist)

People have different opinions on who should provide services in the United States. Who do you think should primarily provide... School education for children?
['Government', 'Private companies/for-profit organisations', 'Non-profit organisations/charities/cooperatives', 'Religious organisations', 'Family, relatives or friends']
Female 793 [0.79319042 0.05800757 0.04035309 0.00630517 0.10214376]
Male 556 [0.79496403 0.05395683 0.0557554  0.00539568 0.08992806]


In [6]:
def make_eval_record(sub_df, cols):
    for c in cols:
        if sub_df[c].nunique(dropna=False) != 1:
            raise ValueError(f"sub_df must have a single value for {c!r}, got {sub_df[c].unique().tolist()}")
    row = sub_df.iloc[0]
    qid = row["qid"]
    q_type = row["type"]
    option_texts = row["option_texts"]
    human_dist = np.stack(sub_df["one_hot"].values).mean(axis=0)

    state_cols = ["c_alphan"] + [c for c in cols if c != "c_alphan"]
    state = "\n".join(f"{PERSONA_NAMES[c]}: {row[c]}" for c in state_cols)
    persona_label = ", ".join(f"{PERSONA_NAMES[c]}={row[c]}" for c in cols) if cols else "All"

    if q_type == "score":
        criteria = option_texts
        target = {str(i): float(p) for i, p in enumerate(human_dist)}
    elif q_type == "choice":
        criteria = {text: None for text in option_texts}
        target = {text: float(p) for text, p in zip(option_texts, human_dist)}
    else:
        raise ValueError(f"{qid}: unexpected question type {q_type!r}")

    return {
        "qid": qid,
        "year": row["year"],
        "v": row["v"],
        "type": q_type,
        "attribute": "+".join(cols) if cols else "none",
        "persona": persona_label,
        "n": len(sub_df),
        "human_dist": human_dist,
        "view": {
            "state": state,
            "questions": {
                qid: {
                    "type": q_type,
                    "instructions": row["question"],
                    "criteria": criteria,
                    "target": target,
                }
            },
        },
    }


def build_persona_dataset(df, attr_sets, min_n=None):
    if isinstance(attr_sets, str):
        attr_sets = [[attr_sets]]
    records = []
    q_groups = list(df.groupby("qid", sort=False))
    for _, q_df in tqdm(q_groups, desc="build_persona_dataset", unit="question"):
        for cols in attr_sets:
            cols = [cols] if isinstance(cols, str) else list(cols)
            valid_df = q_df.dropna(subset=cols) if cols else q_df
            if not cols:
                if min_n is None or len(valid_df) >= min_n:
                    records.append(make_eval_record(valid_df, cols))
                continue
            for _, sub_df in valid_df.groupby(cols, sort=False, observed=True):
                if min_n is None or len(sub_df) >= min_n:
                    records.append(make_eval_record(sub_df, cols))
    return records

In [7]:
test_records = build_persona_dataset(
    test_df,
    attr_sets=["SEX", "AGE"],   # each string is a 1-attribute persona
    min_n=30,
)
pd.DataFrame(test_records)

build_persona_dataset:   0%|          | 0/72 [00:00<?, ?question/s]

,qid,year,v,type,attribute,persona,n,human_dist,view
0,2015:v11,2015,v11,score,SEX,Sex=Male,702,"[0.2037037037037037, 0.4373219373219373, 0.249...","{'state': 'Country: United States Sex: Male', ..."
1,2015:v11,2015,v11,score,SEX,Sex=Female,766,"[0.2349869451697128, 0.41906005221932113, 0.23...",{'state': 'Country: United States Sex: Female'...
2,2015:v11,2015,v11,score,AGE,Age=65+,302,"[0.23841059602649006, 0.48013245033112584, 0.1...","{'state': 'Country: United States Age: 65+', '..."
3,2015:v11,2015,v11,score,AGE,Age=50-64,411,"[0.22871046228710462, 0.4282238442822384, 0.24...","{'state': 'Country: United States Age: 50-64',..."
4,2015:v11,2015,v11,score,AGE,Age=18-29,240,"[0.20833333333333334, 0.3875, 0.2625, 0.133333...","{'state': 'Country: United States Age: 18-29',..."
...,...,...,...,...,...,...,...,...,...
422,2024:v58,2024,v58,choice,SEX,Sex=Female,852,"[0.04460093896713615, 0.06455399061032864, 0.0...",{'state': 'Country: United States Sex: Female'...
423,2024:v58,2024,v58,choice,AGE,Age=30-49,521,"[0.046065259117082535, 0.06333973128598848, 0....","{'state': 'Country: United States Age: 30-49',..."
424,2024:v58,2024,v58,choice,AGE,Age=18-29,207,"[0.07246376811594203, 0.10628019323671498, 0.0...","{'state': 'Country: United States Age: 18-29',..."
425,2024:v58,2024,v58,choice,AGE,Age=65+,380,"[0.042105263157894736, 0.034210526315789476, 0...","{'state': 'Country: United States Age: 65+', '..."


In [8]:
test_records = build_persona_dataset(
    test_df,
    attr_sets='SEX',   # each string is a 1-attribute persona
    min_n=30,
)
pd.DataFrame(test_records)

build_persona_dataset:   0%|          | 0/72 [00:00<?, ?question/s]

,qid,year,v,type,attribute,persona,n,human_dist,view
0,2015:v11,2015,v11,score,SEX,Sex=Male,702,"[0.2037037037037037, 0.4373219373219373, 0.249...","{'state': 'Country: United States Sex: Male', ..."
1,2015:v11,2015,v11,score,SEX,Sex=Female,766,"[0.2349869451697128, 0.41906005221932113, 0.23...",{'state': 'Country: United States Sex: Female'...
2,2015:v29,2015,v29,score,SEX,Sex=Female,462,"[0.5974025974025974, 0.341991341991342, 0.0281...",{'state': 'Country: United States Sex: Female'...
3,2015:v29,2015,v29,score,SEX,Sex=Male,470,"[0.5382978723404256, 0.4021276595744681, 0.031...","{'state': 'Country: United States Sex: Male', ..."
4,2015:v47,2015,v47,score,SEX,Sex=Female,462,"[0.09956709956709957, 0.13852813852813853, 0.2...",{'state': 'Country: United States Sex: Female'...
...,...,...,...,...,...,...,...,...,...
137,2024:v44,2024,v44,score,SEX,Sex=Female,892,"[0.1827354260089686, 0.5627802690582959, 0.181...",{'state': 'Country: United States Sex: Female'...
138,2024:v52,2024,v52,score,SEX,Sex=Male,596,"[0.10738255033557047, 0.2986577181208054, 0.52...","{'state': 'Country: United States Sex: Male', ..."
139,2024:v52,2024,v52,score,SEX,Sex=Female,771,"[0.0959792477302205, 0.2710765239948119, 0.574...",{'state': 'Country: United States Sex: Female'...
140,2024:v58,2024,v58,choice,SEX,Sex=Male,646,"[0.05108359133126935, 0.043343653250773995, 0....","{'state': 'Country: United States Sex: Male', ..."


In [9]:
test_records = build_persona_dataset(
    test_df,
    attr_sets=['SEX'],   # each string is a 1-attribute persona
    min_n=30,
)
pd.DataFrame(test_records)

build_persona_dataset:   0%|          | 0/72 [00:00<?, ?question/s]

,qid,year,v,type,attribute,persona,n,human_dist,view
0,2015:v11,2015,v11,score,SEX,Sex=Male,702,"[0.2037037037037037, 0.4373219373219373, 0.249...","{'state': 'Country: United States Sex: Male', ..."
1,2015:v11,2015,v11,score,SEX,Sex=Female,766,"[0.2349869451697128, 0.41906005221932113, 0.23...",{'state': 'Country: United States Sex: Female'...
2,2015:v29,2015,v29,score,SEX,Sex=Female,462,"[0.5974025974025974, 0.341991341991342, 0.0281...",{'state': 'Country: United States Sex: Female'...
3,2015:v29,2015,v29,score,SEX,Sex=Male,470,"[0.5382978723404256, 0.4021276595744681, 0.031...","{'state': 'Country: United States Sex: Male', ..."
4,2015:v47,2015,v47,score,SEX,Sex=Female,462,"[0.09956709956709957, 0.13852813852813853, 0.2...",{'state': 'Country: United States Sex: Female'...
...,...,...,...,...,...,...,...,...,...
137,2024:v44,2024,v44,score,SEX,Sex=Female,892,"[0.1827354260089686, 0.5627802690582959, 0.181...",{'state': 'Country: United States Sex: Female'...
138,2024:v52,2024,v52,score,SEX,Sex=Male,596,"[0.10738255033557047, 0.2986577181208054, 0.52...","{'state': 'Country: United States Sex: Male', ..."
139,2024:v52,2024,v52,score,SEX,Sex=Female,771,"[0.0959792477302205, 0.2710765239948119, 0.574...",{'state': 'Country: United States Sex: Female'...
140,2024:v58,2024,v58,choice,SEX,Sex=Male,646,"[0.05108359133126935, 0.043343653250773995, 0....","{'state': 'Country: United States Sex: Male', ..."


In [10]:
test_records[0]

{'qid': '2015:v11',
 'year': '2015',
 'v': 'v11',
 'type': 'score',
 'attribute': 'SEX',
 'persona': 'Sex=Male',
 'n': 702,
 'human_dist': array([0.2037037 , 0.43732194, 0.24928775, 0.09259259, 0.01709402]),
 'view': {'state': 'Country: United States\nSex: Male',
  'questions': {'2015:v11': {'type': 'score',
    'instructions': 'Please show how important you personally think the following is in a job. How important is... … a job that involves personal contact with other people',
    'criteria': ['Very important',
     'Important',
     'Neither important nor unimportant',
     'Not important',
     'Not important at all'],
    'target': {'0': 0.2037037037037037,
     '1': 0.4373219373219373,
     '2': 0.2492877492877493,
     '3': 0.09259259259259259,
     '4': 0.017094017094017096}}}}}

In [11]:
@torch.no_grad()
def predict_probs(model, records, batch_size=64):
    model.eval()
    preds = []
    for start in tqdm(range(0, len(records), batch_size), desc="predict", unit="batch"):
        batch = records[start:start + batch_size]
        prepared = [model.prepare(r["view"]) for r in batch]
        logits_batch = model.forward([enc for enc, _ in prepared])
        for question_logits in logits_batch:
            if len(question_logits) != 1:
                raise ValueError(f"Expected 1 question per record, got {len(question_logits)}")
            preds.append(torch.softmax(question_logits[0].float(), dim=-1).cpu().numpy())
    return preds

def consensus(probs):
    num_options = len(probs)
    if num_options <= 1:
        raise ValueError(f"consensus requires at least 2 options, got {num_options}")
    scale_points = np.arange(1, num_options + 1, dtype=float)
    mean_score = float(np.dot(scale_points, probs))
    return float(
        1.0 + np.sum(probs * np.log2(1.0 - np.abs(scale_points - mean_score) / (num_options - 1) + 1e-12))
    )

def evaluate_pair(human_dist, model_dist, q_type):
    if human_dist.shape != model_dist.shape:
        raise ValueError(f"Shape mismatch: {human_dist.shape} != {model_dist.shape}")
    clipped_model = np.clip(model_dist, 1e-12, 1.0)
    nonzero = human_dist > 0
    ce = float(-np.sum(human_dist[nonzero] * np.log2(clipped_model[nonzero])))
    kl = float(np.sum(human_dist[nonzero] * np.log2(human_dist[nonzero] / clipped_model[nonzero])))

    if q_type == "score":
        w1 = float(np.abs(np.cumsum(human_dist)[:-1] - np.cumsum(model_dist)[:-1]).sum() / (len(human_dist) - 1))
        delta_cns = consensus(model_dist) - consensus(human_dist)
    elif q_type == "choice":
        w1 = delta_cns = np.nan
    else:
        raise ValueError(f"Unexpected question type {q_type!r}")
    return {"ce": ce, "dist_kl": kl, "w1": w1, "delta_cns": delta_cns}


def evaluate_records(records, preds):
    if len(records) != len(preds):
        raise ValueError(f"Length mismatch: {len(records)} records vs {len(preds)} predictions")
    rows = []
    for r, model_dist in tqdm(zip(records, preds), total=len(records), desc="evaluate", unit="record"):
        metrics = evaluate_pair(r["human_dist"], model_dist, r["type"])
        rows.append({
            "qid": r["qid"], "year": r["year"], "v": r["v"], "type": r["type"],
            "attribute": r["attribute"], "persona": r["persona"], "n": r["n"],
            **metrics,
        })
    return pd.DataFrame(rows)

# jaredpalmer/kev-0.8b

In [12]:
device = "cuda"
model = KevPointerModel.load("jaredpalmer/kev-0.8b", device)
model.net.head.temperature = 1.0
print(model.describe())

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

{'kind': 'kev', 'base': 'Qwen/Qwen3.5-0.8B-Base', 'mode': 'loaded', 'trainable_params': 524800}


In [ ]:
predict_probs(model,[test_records[0]])

predict:   0%|          | 0/1 [00:00<?, ?batch/s]

[array([0.07194989, 0.2986058 , 0.32024148, 0.19238105, 0.11682171],
       dtype=float32)]

In [14]:
single_persona_attrs = [c for c in persona_cols if c != "c_alphan"]
test_records = build_persona_dataset(test_df, attr_sets=single_persona_attrs, min_n=100)
len(test_records)

build_persona_dataset:   0%|          | 0/72 [00:00<?, ?question/s]

3199

In [15]:
preds = predict_probs(model, test_records, batch_size=64)
results_df = evaluate_records(test_records, preds)

print(results_df[["ce", "dist_kl", "w1", "delta_cns"]].mean())
display(results_df.head(10))

predict:   0%|          | 0/50 [00:00<?, ?batch/s]

evaluate:   0%|          | 0/3199 [00:00<?, ?record/s]

ce           2.873854
dist_kl      1.045691
w1           0.170567
delta_cns    0.094519
dtype: float64


,qid,year,v,type,attribute,persona,n,ce,dist_kl,w1,delta_cns
0,2015:v11,2015,v11,score,SEX,Sex=Male,702,2.314047,0.406811,0.170595,-0.013475
1,2015:v11,2015,v11,score,SEX,Sex=Female,766,2.287970,0.364910,0.165142,-0.030488
2,2015:v11,2015,v11,score,AGE,Age=65+,302,2.068590,0.253702,0.119185,-0.088902
3,2015:v11,2015,v11,score,AGE,Age=50-64,411,2.079205,0.168911,0.110223,-0.053734
4,2015:v11,2015,v11,score,AGE,Age=18-29,240,2.063115,0.109987,0.058840,-0.035909
5,2015:v11,2015,v11,score,AGE,Age=30-49,508,2.042218,0.104638,0.077446,-0.048306
6,2015:v11,2015,v11,score,EDULEVEL,Education=College graduate/some postgrad,439,1.855639,0.035304,0.030068,-0.010894
7,2015:v11,2015,v11,score,EDULEVEL,Education=Upper secondary,768,2.008269,0.076497,0.064988,-0.052582
8,2015:v11,2015,v11,score,EDULEVEL,Education=Short-cycle tertiary,103,1.959436,0.070113,0.030899,0.010522
9,2015:v11,2015,v11,score,EDULEVEL,Education=Less than high school,157,2.513160,0.477783,0.194433,0.029549


In [17]:
display(results_df.groupby(['type'])[["ce", "dist_kl", "w1", "delta_cns"]].mean())

,ce,dist_kl,w1,delta_cns
type,,,,
choice,2.933126,1.688587,NaN,NaN
score,2.864724,0.946659,0.170567,0.094519


train_2022-v4

In [ ]:
device = 'cuda'

model2 = KevPointerModel.load('runs/train_2022-v4/best', device)

In [21]:

preds = predict_probs(model2, test_records, batch_size=64)
results_df = evaluate_records(test_records, preds)

print(results_df[["ce", "dist_kl", "w1", "delta_cns"]].mean())
display(results_df.head(10))

predict:   0%|          | 0/50 [00:00<?, ?batch/s]

evaluate:   0%|          | 0/3199 [00:00<?, ?record/s]

ce           2.327777
dist_kl      0.499614
w1           0.170438
delta_cns   -0.023989
dtype: float64


,qid,year,v,type,attribute,persona,n,ce,dist_kl,w1,delta_cns
0,2015:v11,2015,v11,score,SEX,Sex=Male,702,2.218641,0.311405,0.175367,-0.060296
1,2015:v11,2015,v11,score,SEX,Sex=Female,766,2.039851,0.116791,0.057586,-0.107980
2,2015:v11,2015,v11,score,AGE,Age=65+,302,2.061143,0.246255,0.156550,-0.176325
3,2015:v11,2015,v11,score,AGE,Age=50-64,411,1.988636,0.078343,0.070179,-0.106892
4,2015:v11,2015,v11,score,AGE,Age=18-29,240,2.158242,0.205114,0.131356,-0.093501
5,2015:v11,2015,v11,score,AGE,Age=30-49,508,2.295774,0.358195,0.190669,-0.057532
6,2015:v11,2015,v11,score,EDULEVEL,Education=College graduate/some postgrad,439,2.209158,0.388823,0.188171,-0.093573
7,2015:v11,2015,v11,score,EDULEVEL,Education=Upper secondary,768,2.189115,0.257342,0.154215,-0.067341
8,2015:v11,2015,v11,score,EDULEVEL,Education=Short-cycle tertiary,103,2.184596,0.295272,0.161055,-0.056570
9,2015:v11,2015,v11,score,EDULEVEL,Education=Less than high school,157,2.341354,0.305978,0.176586,-0.028647


In [22]:
display(results_df.groupby(['type'])[["ce", "dist_kl", "w1", "delta_cns"]].mean())

,ce,dist_kl,w1,delta_cns
type,,,,
choice,1.937096,0.692558,NaN,NaN
score,2.387958,0.469893,0.170438,-0.023989
